# Ames Real Estate Valuation: Monotonic XGBoost Model Training & Benchmarking

**Objective**: Benchmark Linear Regression, Random Forest, and Monotonic XGBoost Regressors across 5-Fold Cross Validation ($R^2$, RMSE, MAE) and evaluate Explainable AI (XAI) feature importances.

---

## 1. Environment Setup & Data Pipeline

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from pathlib import Path
from sklearn.model_selection import KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

dataset_path = Path('../data/AmesHousing.csv')
df_raw = pd.read_csv(dataset_path)

base_features = ['Overall Qual', 'Gr Liv Area', 'Garage Cars', 'Full Bath', 'Bedroom AbvGr', 'Year Built']
all_cols = base_features + ['SalePrice']
df_clean = df_raw[(df_raw['Gr Liv Area'] < 4000) & (df_raw['SalePrice'] > 0)].dropna(subset=all_cols).copy()

X = df_clean[base_features].copy()
qual_map = {'Very_Poor': 1, 'Poor': 2, 'Fair': 3, 'Below_Average': 4, 'Average': 5, 'Above_Average': 6, 'Good': 7, 'Very_Good': 8, 'Excellent': 9, 'Very_Excellent': 10}
if X['Overall Qual'].dtype == object:
    X['Overall Qual'] = X['Overall Qual'].map(qual_map).fillna(5)
X['Overall Qual'] = pd.to_numeric(X['Overall Qual'], errors='coerce').fillna(5).astype(float)
X['Qual_Area_Interaction'] = X['Overall Qual'] * X['Gr Liv Area']
y = df_clean['SalePrice']

print(f"Feature Matrix: {X.shape[0]} rows, {X.shape[1]} columns.")

## 2. 5-Fold Cross Validation Benchmark

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Linear Regression (Baseline)": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42),
    "Monotonic XGBoost Regressor": xgb.XGBRegressor(
        n_estimators=220,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        monotone_constraints='(1, 1, 1, 1, 0, 1, 1)',
        random_state=42
    )
}

results = []
for name, model in models.items():
    cv_res = cross_validate(
        model, X, y, cv=kf,
        scoring={'r2': 'r2', 'rmse': 'neg_root_mean_squared_error', 'mae': 'neg_mean_absolute_error'}
    )
    mean_r2 = cv_res['test_r2'].mean()
    mean_rmse = -cv_res['test_rmse'].mean()
    mean_mae = -cv_res['test_mae'].mean()
    results.append({
        "Model Paradigm": name,
        "R^2 Score": round(mean_r2, 4),
        "RMSE ($)": f"${mean_rmse:,.2f}",
        "MAE ($)": f"${mean_mae:,.2f}"
    })

pd.DataFrame(results)

## 3. Production Monotonic XGBoost Feature Importances (Gain)

In [ ]:
prod_model = models["Monotonic XGBoost Regressor"]
prod_model.fit(X, y)

importances = prod_model.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': X.columns,
    'Importance (%)': (importances * 100).round(2)
}).sort_values('Importance (%)', ascending=False)

feat_imp